<a href="https://colab.research.google.com/github/harveytheato/Harveytheato-DataScience-GenAI-Submissions/blob/main/6_02_DNN_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=1xqQczl0FG-qtNA2_WQYuWePW9oU8irqJ)

# 6.02 Dense Neural Network (with PyTorch)
This will expand on our logistic regression example and take us through building our first neural network. If you haven't already, be sure to check (and if neccessary) switch to GPU processing by clicking Runtime > Change runtime type and selecting GPU. We can test this has worked with the following code:

In [ ]:
import torch

# Check for GPU availability
print("Num GPUs Available: ", torch.cuda.device_count())

Num GPUs Available:  1


Hopefully your code shows you have 1 GPU available! Next let's get some data. We'll start with another in-built dataset:

In [ ]:
# upload an in-built Python (OK semi-in-built) dataset
from sklearn.datasets import load_diabetes

import pandas as pd
import numpy as np

# import the data
data = load_diabetes()
data

{'data': array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
          0.01990749, -0.01764613],
        [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
         -0.06833155, -0.09220405],
        [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
          0.00286131, -0.02593034],
        ...,
        [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
         -0.04688253,  0.01549073],
        [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
          0.04452873, -0.02593034],
        [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
         -0.00422151,  0.00306441]], shape=(442, 10)),
 'target': array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
         69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
         68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
         87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
        259.,  53., 190., 142.,  75., 142.

We are working on a regression problem, with "structured" data which has already been cleaned and normalised. We can skip the usual cleaning/engineering steps. However, we do need to get the data into PyTorch:

In [ ]:
# Convert data to PyTorch tensors
X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1) # Reshape y to be a column vector

Now our data is stored in tensors we can do train/test splitting as before (in fact we can use sklearn as before):

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch.Size([353, 10]) torch.Size([353, 1])
torch.Size([89, 10]) torch.Size([89, 1])


Now we can set up our batches for training. As we have a nice round 400 let's go with batches of 50 (8 batches in total). We'll also seperate the features and labels:

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

Now its time to build our model. We'll keep it simple ... a model with an input layer of 10 features and then 2x _Dense_ (fully connected) layers each with 5 neurons and ReLU activation. Our output layer will be size=1 given this is a regression problem and we want a single value output per prediction.

This will be easier to understand if you have read through the logistic regression tutorial.

In [ ]:
import torch
import torch.nn as nn

# Define the model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer will be a linear layer that has 5x neurons
            # (5x sets of linear regression)
            # the layer takes the 10 features as input (i.e. 10, 5)
            nn.Linear(10, 5),

            nn.ReLU(), # ReLU activation

            # second linear layer again has 5 neurons
            # this time taking the input as the output of the last layer
            # (which had 5x neurons)
            nn.Linear(5, 5),

            nn.ReLU(), # ReLU again

            # last linear layer takes the output from the previous 5 neurons
            # this time its a single output with no activation
            # i.e. this is the predicitons (regression)
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers

As before we need to create a model object, specify the loss (criterion) and an optimiser (which we cover next week):

In [ ]:
import torch.optim as optim

# Initialize the model, loss function, and optimizer
model = DiabetesModel()
criterion = nn.MSELoss() # MSE loss function
optimiser = optim.Adam(model.parameters(), lr=0.001)

Now we can train the model. Again, the logistic regression tutorial (6.01) may help you undertstand this:

In [ ]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop (example - you'll likely want to add more epochs)
epochs = 100 # 100 epochs

for epoch in range(epochs):
  # use the train_loader to pass the inputs (x) and targets (y)
  for inputs, targets in train_loader:
    # pass to the GPU (hopefully)
    inputs, targets = inputs.to(device), targets.to(device)

    # pass model to GPU as well
    model.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradiants
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropogate the loss (next week)
    optimiser.step() # # update the parameters based on this round of training

  # every 10 steps we will print out the current loss
    if (epoch+1) % 10 == 0: # modular arithmetic
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/100], Loss: 29538.4512
Epoch [10/100], Loss: 33246.8633
Epoch [10/100], Loss: 33234.5586
Epoch [10/100], Loss: 25375.6387
Epoch [10/100], Loss: 35956.9766
Epoch [10/100], Loss: 25024.3887
Epoch [10/100], Loss: 25211.1289
Epoch [10/100], Loss: 22746.2852
Epoch [20/100], Loss: 31993.0898
Epoch [20/100], Loss: 32114.4395
Epoch [20/100], Loss: 24379.459
Epoch [20/100], Loss: 31707.5352
Epoch [20/100], Loss: 23582.3691
Epoch [20/100], Loss: 28440.5391
Epoch [20/100], Loss: 33206.0352
Epoch [20/100], Loss: 43520.4844
Epoch [30/100], Loss: 30167.6934
Epoch [30/100], Loss: 29593.4824
Epoch [30/100], Loss: 31597.209
Epoch [30/100], Loss: 22614.1094
Epoch [30/100], Loss: 33947.1641
Epoch [30/100], Loss: 29556.8125
Epoch [30/100], Loss: 28195.3047
Epoch [30/100], Loss: 13201.709
Epoch [40/100], Loss: 23414.2695
Epoch [40/100], Loss: 32997.4141
Epoch [40/100], Loss: 32956.5781
Epoch [40/100], Loss: 25668.7793
Epoch [40/100], Loss: 28715.4844
Epoch [40/100], Loss: 25488.9785
Epoch [40/100

We can see loss is significantly lower at the end than it was at the start. However, it is also bouncing around a little still which suggests the model needs more training (100 epochs is not a lot in deep learning terms). However, let's evaluate as before:

In [ ]:
# Evaluation (example)
model.eval() # testing mode
mse_values = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets) # calcualte mse for the batch
        mse_values.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse = np.mean(mse_values)
print(f"Average MSE on test set: {avg_mse}")

Average MSE on test set: 19265.01953125


MSE looks expected given training (no obvious sign of overfitting). However, we probably can get better results with tuning and more epochs.

Let's run the loop again a little differently to collect the predicted values (y_hat) and actuals (y) and add them to a dataset for comparions:

In [ ]:
# Evaluation
model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Create DataFrame
results_df = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
results_df

,Predicted,Actual
0,27.821922,219.0
1,25.606659,70.0
2,27.214096,202.0
3,33.265240,230.0
4,26.223549,111.0
...,...,...
84,23.513599,153.0
85,21.476261,98.0
86,19.407936,37.0
87,20.237206,63.0


Side-by-side, they don't look great. Can you improve them?

<br><br>

## EXERCISE #1
Try increasing the number of epochs to 1,000 (when the model is fairly well trained then the results printed for each 10x epochs will be fairly stable and not change much). Does this give better results?

<br><br>

## EXERCISE #2 (optional)
Try experimenting with the architecture (number of neurons and/or number of layers). Can we reach an optimal architecture?

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Define the model architecture
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(10, 5),
            nn.ReLU(),
            nn.Linear(5, 5),
            nn.ReLU(),
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x)

# Re-initialize the model, criterion, and optimizer
model = DiabetesModel()
criterion = nn.MSELoss()
optimiser = optim.Adam(model.parameters(), lr=0.001)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Training for 1000 epochs
epochs = 1000

for epoch in range(epochs):
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        model.train()
        optimiser.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimiser.step()

    # Print loss every 100 epochs to keep the output clean
    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [100/1000], Loss: 5251.3828
Epoch [200/1000], Loss: 1482.762
Epoch [300/1000], Loss: 2267.5479
Epoch [400/1000], Loss: 4843.9688
Epoch [500/1000], Loss: 743.3691
Epoch [600/1000], Loss: 497.2598
Epoch [700/1000], Loss: 3407.4453
Epoch [800/1000], Loss: 2450.3359
Epoch [900/1000], Loss: 4015.4365
Epoch [1000/1000], Loss: 536.0775


In [ ]:
y_train.var().item()

6093.66064453125

Increasing the number of epochs from 100 to 1,000 improved the test set mean squared error from 19,265 to 6,094, so more training did give better results. However, 6,094 is close to the variance of the target variable (roughly 5,900), which is what a model would score by always predicting the average value and ignoring the features. This suggests the extra epochs mainly helped the model’s predictions reach the correct scale of the target (values of roughly 25 to 346) rather than learning a strong relationship between the features and disease progression. The training loss printed during training also fluctuated heavily because it was taken from a single batch, so the test set score is a more reliable measure of performance. Further improvement would probably need changes such as standardising the target variable or adjusting the architecture

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Standardise the target using training data only
y_mean, y_std = y_train.mean(), y_train.std()
y_train_s = (y_train - y_mean) / y_std

train_loader_s = DataLoader(TensorDataset(X_train, y_train_s), batch_size=50, shuffle=True)

def build_model(hidden_sizes):
    layers = []
    in_features = 10
    for size in hidden_sizes:
        layers += [nn.Linear(in_features, size), nn.ReLU()]
        in_features = size
    layers.append(nn.Linear(in_features, 1))
    return nn.Sequential(*layers)

def train_and_evaluate(hidden_sizes, epochs=500, lr=0.001, seed=42):
    torch.manual_seed(seed)  # same starting conditions for a fair comparison
    model = build_model(hidden_sizes).to(device)
    criterion = nn.MSELoss()
    optimiser = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        for inputs, targets in train_loader_s:
            inputs, targets = inputs.to(device), targets.to(device)
            optimiser.zero_grad()
            loss = criterion(model(inputs), targets)
            loss.backward()
            optimiser.step()

    # Evaluate on the whole test set at once, converted back to original units
    model.eval()
    with torch.no_grad():
        preds = model(X_test.to(device)).cpu() * y_std + y_mean
        test_mse = criterion(preds, y_test).item()
    return test_mse

# Architectures to compare (each number is the neurons in one hidden layer)
architectures = {
    "5-5 (original)": [5, 5],
    "10": [10],
    "10-10": [10, 10],
    "32-32": [32, 32],
    "64-32": [64, 32],
    "64-32-16": [64, 32, 16],
}

baseline = ((y_test - y_train.mean()) ** 2).mean().item()
print(f"Baseline (always predict the training average): {baseline:.1f}\n")

results = {}
for name, sizes in architectures.items():
    results[name] = train_and_evaluate(sizes)
    print(f"{name:16s} test mean squared error: {results[name]:.1f}")

best = min(results, key=results.get)
print(f"\nBest architecture: {best} ({results[best]:.1f})")

Baseline (always predict the training average): 5361.5

5-5 (original)   test mean squared error: 2826.6
10               test mean squared error: 2863.3
10-10            test mean squared error: 2837.0
32-32            test mean squared error: 2799.7
64-32            test mean squared error: 2563.1
64-32-16         test mean squared error: 2549.1

Best architecture: 64-32-16 (2549.1)


Exercise 2: I compared six architectures after standardising the target variable, training each for 500 epochs with the same random seed. All beat the baseline of always predicting the training average (test mean squared error 5,361.5). The original 5-5 network scored 2,826.6, while the best, 64-32-16, scored 2,549.1, a modest improvement of about 10%. The 64-32 network (2,563.1) was almost identical, so adding a third layer gave no meaningful benefit. The smaller networks (5-5, 10, 10-10, 32-32) were all within about 2% of one another, suggesting that beyond a certain size, extra neurons help little. Standardising the target produced a larger improvement than any architecture change (6,094 to 2,827 for the original network). Because the test set has only 89 observations and I used a single seed, I cannot claim 64-32-16 is truly optimal, only the best of the architectures tried. Averaging over several seeds or using cross-validation would give a more reliable comparison.
